In [1]:
import torch
import torchvision.transforms as transforms
import train_load_datasets_resnet as tr
import numpy as np
from torch.utils.data import Subset

In [2]:
def get_labels_array(ds):
    """
    Return a 1D numpy array of labels for any torch Dataset, possibly nested in Subset(s).
    Handles datasets exposing `.labels` (e.g., MedMNIST) or `.targets` (e.g., torchvision).
    As a last resort, indexes ds[i][1] for all i (slow, but generic).
    """
    # Unwrap nested Subset(s) and compose indices
    idx = None
    while isinstance(ds, Subset):
        if idx is None:
            idx = np.array(ds.indices)
        else:
            # compose indices if we have Subset(Subset(base))
            idx = np.array(ds.indices)[idx]
        ds = ds.dataset

    # Now ds is the base dataset
    if hasattr(ds, "labels"):
        labels = np.array(ds.labels)
    elif hasattr(ds, "targets"):
        labels = np.array(ds.targets)
    else:
        # Generic (slow) fallback: index the dataset to get labels
        labels = np.array([ds[i][1] for i in range(len(ds))])

    # Squeeze to 1D in case of shape (N,1)
    labels = np.asarray(labels).squeeze()

    # Apply subset indices if any
    if idx is not None:
        labels = labels[idx]

    return labels

def get_class_counts(ds):
    labels_arr = get_labels_array(ds)
    _, counts = np.unique(labels_arr, return_counts=True)
    return counts


In [3]:
def evenness_stats(counts):
    counts = np.asarray(counts, dtype=float)
    total = counts.sum()
    S = len(counts)
    if total == 0 or S == 0:
        return {"S": S, "J": 0.0, "Neff": 0.0, "Neff_over_S": 0.0,
                "TV": 0.0, "IR": np.nan, "p_max": 0.0, "p_min": 0.0}
    p = counts / total
    p_nz = p[p > 0]
    H = -(p_nz * np.log(p_nz)).sum()
    J = H / np.log(S) if S > 1 else 1.0
    Neff = float(np.exp(H))
    TV = 0.5 * np.abs(p - 1.0 / S).sum()
    p_min = float(p_nz.min()) if p_nz.size else 0.0
    IR = float(p.max() / p_min) if p_min > 0 else np.inf
    return {
        "S": S,
        "J": float(J),
        "Neff": Neff,
        "Neff_over_S": float(Neff / S),
        "TV": float(TV),
        "IR": IR,
        "p_max": float(p.max()),
        "p_min": p_min,
    }

def format_stats(st):
    S = int(round(st['S']))
    return (f"J={st['J']:.3f}, Neff={st['Neff']:.2f}/{S} ({st['Neff_over_S']:.1%}), "
            f"TV={st['TV']:.3f}, IR={st['IR']:.2f}, pmin={st['p_min']:.1%}, pmax={st['p_max']:.1%}")


def aggregate_over_loaders(loaders, get_class_counts_fn):
    sizes = [len(ld.dataset) for ld in loaders]
    stats_list = [evenness_stats(get_class_counts_fn(ld.dataset)) for ld in loaders]
    keys = ["J","Neff","Neff_over_S","TV","IR","p_max","p_min","S"]
    mean_stats = {k: float(np.mean([d[k] for d in stats_list])) for k in keys}
    mean_size = int(round(np.mean(sizes)))
    return mean_size, mean_stats

In [ ]:
flags = ['breastmnist', 'organamnist', 'pneumoniamnist', 'dermamnist-e', 'octmnist', 'pathmnist', 'bloodmnist', 'tissuemnist']
colors = [False, False, False, True, False, True, True, False]  # Colors for the flags
device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
size = 224  # Image size for the models
batch_size = 4000  # Batch size for the DataLoader
model_global_perfs = {}

for flag, color in zip(flags, colors):
    if color is True:
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[.5, .5, .5], std=[.5, .5, .5])
        ])
    else:
        # For grayscale images, repeat the single channel to make it compatible with ResNet
        # ResNet expects 3 channels, so we repeat the single channel image
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[.5], std=[.5]),
            transforms.Lambda(lambda x: x.repeat(3, 1, 1))
        ])
    
    [train_dataset, calibration_dataset, test_dataset], [train_loader, calibration_loader, test_loader], info = tr.load_datasets(flag, color, size, transform, batch_size)
    train_loaders, val_loaders = tr.CV_train_val_loaders(None, train_dataset, batch_size=batch_size)
    print(f"{flag}:")


   
    # Train / Val (aggregate across folds)
    mean_train_size, train_stats = aggregate_over_loaders(train_loaders, get_class_counts)
    mean_val_size,   val_stats   = aggregate_over_loaders(val_loaders,   get_class_counts)



    calib_size = len(calibration_dataset)
    test_size = len(test_dataset)

    train_equitabilities = [evenness_stats(get_class_counts(loader.dataset)) for loader in train_loaders]
    val_equitabilities = [evenness_stats(get_class_counts(loader.dataset)) for loader in val_loaders]


    # Calibration set
    calib_labels = get_labels_array(calibration_dataset)
    calib_class_counts = np.unique(calib_labels, return_counts=True)[1]

    # test_dataset is a base medmnist dataset (not a Subset)
    test_labels = np.array(test_dataset.labels).flatten()
    test_class_counts = np.unique(test_labels, return_counts=True)[1]

    # Calibration / Test (single dataset each)
    calib_stats = evenness_stats(calib_class_counts)
    test_stats  = evenness_stats(test_class_counts)

    print(f"  Train  mean N={mean_train_size} ({len(train_loaders)} folds): {format_stats(train_stats)}")
    print(f"  Val    mean N={mean_val_size} ({len(val_loaders)} folds):   {format_stats(val_stats)}")
    print(f"  Calibration N={calib_size}: {format_stats(calib_stats)}")
    print(f"  Test        N={test_size}: {format_stats(test_stats)}")

Training dataset size: 499
Calibration dataset size: 125
breastmnist:
  Train  mean N=399 (5 folds): J=0.842, Neff=1.79/2 (89.6%), TV=0.229, IR=2.70, pmin=27.1%, pmax=72.9%
  Val    mean N=100 (5 folds):   J=0.842, Neff=1.79/2 (89.6%), TV=0.229, IR=2.70, pmin=27.1%, pmax=72.9%
  Calibration N=125: J=0.833, Neff=1.78/2 (89.1%), TV=0.236, IR=2.79, pmin=26.4%, pmax=73.6%
  Test         N=156: J=0.840, Neff=1.79/2 (89.5%), TV=0.231, IR=2.71, pmin=26.9%, pmax=73.1%
Training dataset size: 32841
Calibration dataset size: 8211
organamnist:
  Train  mean N=26273 (5 folds): J=0.958, Neff=9.94/11 (90.3%), TV=0.187, IR=4.44, pmin=3.9%, pmax=17.4%
  Val    mean N=6568 (5 folds):   J=0.958, Neff=9.94/11 (90.3%), TV=0.187, IR=4.44, pmin=3.9%, pmax=17.4%
  Calibration N=8211: J=0.954, Neff=9.85/11 (89.5%), TV=0.196, IR=5.01, pmin=3.6%, pmax=17.9%
  Test         N=17778: J=0.960, Neff=10.00/11 (90.9%), TV=0.173, IR=4.19, pmin=4.4%, pmax=18.5%
Training dataset size: 4185
Calibration dataset size: 1047
p

Training dataset size: 499
Calibration dataset size: 125
breastmnist:
  Train  mean N=399 (5 folds): J=0.842, Neff=1.79/2 (89.6%), TV=0.229, IR=2.70, pmin=27.1%, pmax=72.9%
  Val    mean N=100 (5 folds):   J=0.842, Neff=1.79/2 (89.6%), TV=0.229, IR=2.70, pmin=27.1%, pmax=72.9%
  Calibration N=125: J=0.833, Neff=1.78/2 (89.1%), TV=0.236, IR=2.79, pmin=26.4%, pmax=73.6%
  Test         N=156: J=0.840, Neff=1.79/2 (89.5%), TV=0.231, IR=2.71, pmin=26.9%, pmax=73.1%
Training dataset size: 32841
Calibration dataset size: 8211
organamnist:
  Train  mean N=26273 (5 folds): J=0.958, Neff=9.94/11 (90.3%), TV=0.187, IR=4.44, pmin=3.9%, pmax=17.4%
  Val    mean N=6568 (5 folds):   J=0.958, Neff=9.94/11 (90.3%), TV=0.187, IR=4.44, pmin=3.9%, pmax=17.4%
  Calibration N=8211: J=0.954, Neff=9.85/11 (89.5%), TV=0.196, IR=5.01, pmin=3.6%, pmax=17.9%
  Test         N=17778: J=0.960, Neff=10.00/11 (90.9%), TV=0.173, IR=4.19, pmin=4.4%, pmax=18.5%
Training dataset size: 4185
Calibration dataset size: 1047
pneumoniamnist:
  Train  mean N=3348 (5 folds): J=0.826, Neff=1.77/2 (88.6%), TV=0.241, IR=2.86, pmin=25.9%, pmax=74.1%
  Val    mean N=837 (5 folds):   J=0.826, Neff=1.77/2 (88.6%), TV=0.241, IR=2.86, pmin=25.9%, pmax=74.1%
  Calibration N=1047: J=0.815, Neff=1.76/2 (87.9%), TV=0.248, IR=2.97, pmin=25.2%, pmax=74.8%
  Test         N=624: J=0.954, Neff=1.94/2 (96.9%), TV=0.125, IR=1.67, pmin=37.5%, pmax=62.5%
Training dataset size: 8166
Calibration dataset size: 2042
dermamnist-e:
  Train  mean N=6533 (5 folds): J=0.580, Neff=3.09/7 (44.1%), TV=0.529, IR=61.63, pmin=1.1%, pmax=67.2%
  Val    mean N=1633 (5 folds):   J=0.580, Neff=3.09/7 (44.1%), TV=0.529, IR=61.66, pmin=1.1%, pmax=67.2%
  Calibration N=2042: J=0.590, Neff=3.15/7 (45.0%), TV=0.515, IR=58.39, pmin=1.1%, pmax=65.8%
  Test         N=1511: J=0.665, Neff=3.65/7 (52.1%), TV=0.459, IR=25.94, pmin=2.3%, pmax=60.1%
Training dataset size: 86647
Calibration dataset size: 21662
octmnist:
  Train  mean N=69318 (5 folds): J=0.836, Neff=3.19/4 (79.7%), TV=0.315, IR=5.91, pmin=8.0%, pmax=47.3%
  Val    mean N=17329 (5 folds):   J=0.836, Neff=3.19/4 (79.7%), TV=0.315, IR=5.91, pmin=8.0%, pmax=47.3%
  Calibration N=21662: J=0.835, Neff=3.18/4 (79.6%), TV=0.317, IR=6.05, pmin=7.8%, pmax=47.1%
  Test         N=1000: J=1.000, Neff=4.00/4 (100.0%), TV=0.000, IR=1.00, pmin=25.0%, pmax=25.0%
Training dataset size: 80000
Calibration dataset size: 20000
pathmnist:
  Train  mean N=64000 (5 folds): J=0.994, Neff=8.89/9 (98.8%), TV=0.065, IR=1.62, pmin=8.8%, pmax=14.3%
  Val    mean N=16000 (5 folds):   J=0.994, Neff=8.89/9 (98.8%), TV=0.065, IR=1.62, pmin=8.8%, pmax=14.3%
  Calibration N=20000: J=0.994, Neff=8.88/9 (98.6%), TV=0.065, IR=1.71, pmin=8.4%, pmax=14.4%
  Test         N=7180: J=0.961, Neff=8.26/9 (91.8%), TV=0.176, IR=3.95, pmin=4.7%, pmax=18.6%
Training dataset size: 10936
Calibration dataset size: 2735
bloodmnist:
  Train  mean N=8749 (5 folds): J=0.964, Neff=7.42/8 (92.8%), TV=0.181, IR=2.68, pmin=7.2%, pmax=19.2%
  Val    mean N=2187 (5 folds):   J=0.964, Neff=7.42/8 (92.8%), TV=0.181, IR=2.68, pmin=7.2%, pmax=19.2%
  Calibration N=2735: J=0.959, Neff=7.35/8 (91.9%), TV=0.194, IR=3.12, pmin=6.6%, pmax=20.7%
  Test         N=3421: J=0.963, Neff=7.41/8 (92.6%), TV=0.184, IR=2.74, pmin=7.1%, pmax=19.5%
Training dataset size: 151284
Calibration dataset size: 37822
tissuemnist:
  Train  mean N=121027 (5 folds): J=0.868, Neff=6.07/8 (75.9%), TV=0.331, IR=9.07, pmin=3.5%, pmax=32.1%
  Val    mean N=30257 (5 folds):   J=0.868, Neff=6.07/8 (75.9%), TV=0.331, IR=9.07, pmin=3.5%, pmax=32.1%
  Calibration N=37822: J=0.867, Neff=6.07/8 (75.9%), TV=0.333, IR=8.96, pmin=3.6%, pmax=32.1%
  Test         N=47280: J=0.868, Neff=6.07/8 (75.9%), TV=0.331, IR=9.04, pmin=3.5%, pmax=32.1%